In [1]:
!nvidia-smi

Thu Oct  1 03:00:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from datetime import datetime

print("Fecha y hora:", datetime.now().strftime("%d/%m/%Y %H:%M:%S"))
!nvidia-smi

Fecha y hora: 01/10/2026 03:01:32
Thu Oct  1 03:01:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-------------

In [3]:
import cupy as cp

print("CuPy:", cp.__version__)
print("GPU:", cp.cuda.runtime.getDeviceProperties(0)["name"].decode())
print("CUDA runtime:", cp.cuda.runtime.runtimeGetVersion())

CuPy: 14.0.1
GPU: Tesla T4
CUDA runtime: 12090


In [4]:
import numpy as np
import cupy as cp
import time

print("NumPy:", np.__version__)
print("CuPy:", cp.__version__)

NumPy: 2.1.3
CuPy: 14.0.1


In [5]:
import numpy as np
import time

def medir_cpu(n):
    A = np.random.rand(n, n).astype(np.float32)
    B = np.random.rand(n, n).astype(np.float32)

    inicio = time.perf_counter()
    C = A @ B
    fin = time.perf_counter()

    return fin - inicio

In [6]:
import cupy as cp
import time

def medir_gpu(n):
    A = cp.random.rand(n, n, dtype=cp.float32)
    B = cp.random.rand(n, n, dtype=cp.float32)

    cp.cuda.Stream.null.synchronize()

    inicio = time.perf_counter()
    C = A @ B
    cp.cuda.Stream.null.synchronize()
    fin = time.perf_counter()

    return fin - inicio

In [7]:
tamanos = [256, 1024, 4096]

resultados = []

for n in tamanos:
    tiempo_cpu = medir_cpu(n)
    tiempo_gpu = medir_gpu(n)
    speedup = tiempo_cpu / tiempo_gpu

    resultados.append({
        "n": n,
        "CPU NumPy (s)": tiempo_cpu,
        "GPU CuPy (s)": tiempo_gpu,
        "Speedup CPU/GPU": speedup
    })

resultados


[{'n': 256,
  'CPU NumPy (s)': 0.0037246570000206702,
  'GPU CuPy (s)': 0.17397127599997475,
  'Speedup CPU/GPU': 0.021409609020866242},
 {'n': 1024,
  'CPU NumPy (s)': 0.018737957999974242,
  'GPU CuPy (s)': 0.0012345740000228034,
  'Speedup CPU/GPU': 15.177671002004043},
 {'n': 4096,
  'CPU NumPy (s)': 1.2060218230000146,
  'GPU CuPy (s)': 0.04783450200000061,
  'Speedup CPU/GPU': 25.212383793605696}]

In [8]:
def medir_gpu_sin_sync(n):
    A = cp.random.rand(n, n, dtype=cp.float32)
    B = cp.random.rand(n, n, dtype=cp.float32)

    cp.cuda.Stream.null.synchronize()

    inicio = time.perf_counter()
    C = A @ B
    fin = time.perf_counter()

    return fin - inicio

In [9]:
resultados_sin_sync = []

for n in tamanos:
    tiempo = medir_gpu_sin_sync(n)

    resultados_sin_sync.append({
        "n": n,
        "GPU CuPy sin synchronize (s)": tiempo
    })

resultados_sin_sync

[{'n': 256, 'GPU CuPy sin synchronize (s)': 0.0011533269999972617},
 {'n': 1024, 'GPU CuPy sin synchronize (s)': 0.00015671100004510663},
 {'n': 4096, 'GPU CuPy sin synchronize (s)': 0.0019401690000222516}]

In [10]:
import pandas as pd

tabla = pd.DataFrame({
    "n": [256, 1024, 4096],
    "CPU NumPy (s)": [
        0.0037246570000206702,
        0.018737957999974242,
        1.2060218230000146
    ],
    "GPU CuPy con sync (s)": [
        0.17397127599997475,
        0.0012345740000228034,
        0.04783450200000061
    ],
    "GPU CuPy sin sync (s)": [
        0.0011533269999972617,
        0.00015671100004510663,
        0.0019401690000222516
    ],
    "Speedup CPU/GPU": [
        0.021409609020866242,
        15.177671002004043,
        25.212383793605696
    ]
})

tabla

,n,CPU NumPy (s),GPU CuPy con sync (s),GPU CuPy sin sync (s),Speedup CPU/GPU
0,256,0.003725,0.173971,0.001153,0.021410
1,1024,0.018738,0.001235,0.000157,15.177671
2,4096,1.206022,0.047835,0.001940,25.212384


## Resultados CPU vs GPU

La comparación se realizó mediante multiplicación de matrices utilizando NumPy en CPU y CuPy en GPU. Para la medición de GPU con sincronización se utilizó `cp.cuda.Stream.null.synchronize()` antes de detener el cronómetro, con el objetivo de esperar a que la operación terminara realmente.

La medición sin sincronización se realizó para demostrar el comportamiento asíncrono de las operaciones en GPU y mostrar por qué medir únicamente el tiempo observado por el CPU puede producir resultados que no representan el tiempo real de ejecución.